<a href="https://colab.research.google.com/github/Aditi0912dec/LFBC/blob/main/log_of_odds_vs_log_gm_vs_voting_vs_average.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

KALI

In [ ]:
import math

In [ ]:
import sklearn
import tensorflow as tf
from sklearn.datasets import load_digits
from sklearn.preprocessing import LabelBinarizer
from sklearn.model_selection import KFold, train_test_split
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import SGD
from sklearn.neural_network import MLPClassifier
from keras import models
from tensorflow.keras.utils import to_categorical
import time
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Flatten
from tensorflow.keras.models import Sequential
from tensorflow.keras.datasets import mnist
from sklearn.model_selection import train_test_split
import csv
import os
import random
from keras.datasets import fashion_mnist
from sklearn.utils import shuffle

from sklearn.metrics import precision_score,recall_score,accuracy_score

In [ ]:
!pip install pyyaml h5py  # Required to save models in HDF5 format

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
# file CNN_model_define
def get_CNN_model():
  model = Sequential()
  model.add(Conv2D(32, (5, 5), activation='relu', kernel_initializer='he_uniform', input_shape=(28, 28, 1)))

  model.add(MaxPooling2D((2,2)))
  model.add(Conv2D(16, (5, 5), activation='relu', kernel_initializer='he_uniform'))

  model.add(MaxPooling2D((2,2)))
  model.add(Flatten())
  #model.add(Dense(1024, activation='relu', kernel_initializer='he_uniform'))
 # model.add(Dense(512, activation='relu', kernel_initializer='he_uniform'))
  model.add(Dense(2, activation='softmax'))
	# compile model
	#opt = SGD(learning_rate=0.01, momentum=0.9)
  model.compile(optimizer='SGD', loss='categorical_crossentropy', metrics=['accuracy'])
  return model



# read and write

In [ ]:

# file key_ininialize
def initialize_key(device_list,client_clf_list,beta_value):
  for i in range(0,len(device_list)):
    for j in range(0,beta_value):
      device_list[i].key.append(client_clf_list[i][j])
  return device_list

# Load MNIST digits

In [ ]:
# file  get_dataset_labels
def load_dataset():
  print("data loading")
  (X,y),( X_test,y_test) = mnist.load_data()
  print(X.shape,y.shape,X_test.shape,y_test.shape)
  return X,y,X_test,y_test


def data_processing(X):
  # pixel_value in [0,1]
  print("data processing")
  #X = X.reshape((X.shape[0],-1))# converting to 1D array
  X = X/255# normalize the input
  print(X.shape)

  return X


def get_categories(y):
    labels=np.unique(y)
    print("number of classes in the dataset: "+str(len(labels)))
    print("classes in the dataset: "+str(labels))

    return labels

# data distribution

In [ ]:
# get_distribution
import random
import math
def get_datamap(X,y,labels):
  print("split data as per y label")
  digit_map = {}


  for i in range(len(labels)):
    indices = (y==i)
    Xi = X[indices]
    yi = y[indices]
    digit_map[i] = (Xi,yi)

  for a in range(10):
    Xa,ya = digit_map[a]
    print (Xa.shape, ya.shape)

  return digit_map


def noniid_distribuition(digit_map,device_list,label,beta_value):
  print("In non iid distribution part 1")
  device_index = 0
  partition_per_class=math.floor((len(device_list)*beta_value)/len(label))
  not_avail=set()
  idx=set(np.arange(0,len(device_list)))
  # getting the keys executing for the first time
  for i in range(0,len(digit_map)):
    print(f"dataset: {i}")
    m=0
    Xi,yi = digit_map[i]
    #print(Xi.shape,yi.shape)
    if(i>=0):
      for d in idx:
        if len(device_list[d].digit_data.keys())==beta_value:
          not_avail.add(d)

    idx=idx.difference(not_avail)
    if(len(idx)<partition_per_class):
      partition_per_class=len(idx)
    if len(idx)==1:
      break
    sel_device_index=random.sample(list(idx),partition_per_class)
    kf = KFold(n_splits=partition_per_class,shuffle=True)
    for train_indices, test_indices in kf.split(Xi):

      Xa,ya= Xi[test_indices],yi[test_indices]
      device_list[sel_device_index[m]].add_data(digit=i,X=Xa,y=ya)
      m=m+1

################################################
  class_label=list(label)
  print("In non iid distribution part 2")
  if len(idx)==1:
    digit_choice=[x for x in label if x not in set(device_list[i].digit_data.keys())]
    m=random.choice(digit_choice)
    index_0=next(iter(idx))
    device_list[index_0].add_data(digit=m,X=Xi,y=yi)
   # device_list[index_0].add_key(digit=m)
  else:

    for i in idx:
      while len(device_list[i].digit_data.keys())<beta_value:
        digit_choice=[x for x in label if x not in set(device_list[i].digit_data.keys())]
        m=random.choice(digit_choice)
        Xi,yi=digit_map[m]
        kf = KFold(n_splits=partition_per_class,shuffle=True)
        for train_indices, test_indices in kf.split(Xi):
          #device_list[i].add_data(digit=m,X=Xi[test_indices])
          device_list[i].add_data(digit=m,X=Xi,y=yi)
         # device_list[i].add_key(digit=m)
          break

######################################################
  #print("device keys")

  return device_list
#######################################################



In [ ]:
# file client_key_mapping
def client_key_map(device_list):
  print("get the client key map")
  client_key_map=[]

  for i in range(0,len(device_list)):
      client_key_map.append(device_list[i].key)
  print(client_key_map)
  return client_key_map

# get class

In [ ]:
# file class_module
class MyDigitDevice:

  def __init__(self,id=None):
    self.id = id
    print("device id:" +str(id))
  #  self.key=[]
    self.digit_data = {}
    self.digit_clf = {}
    self.digit_dataset={}
  #  self.digit_csv={}

# how to get y?? or y is the digit

  def add_data(self,X=None,y=None,digit=None):
     self.digit_data[digit]=(X,y)

    #self.digit_data[digit] = (X,y)

  def get_training_dataset(self,sel_digit=None):
    print("in get dataset")
    Xcat_list = []
    ycat_list = []
    X_sub=[]
    y_sub=[]

    if sel_digit in self.digit_data.keys():
        client_label=list(self.digit_data.keys())
        for i in client_label:
          Xj, yj = self.digit_data[i]# 10 parts of data with each client
       #   print(Xj.shape)
       #   print(yj)
       #  # Xj=Xj.reshape((Xj.shape[0],28,28))
          #print(i)
          Xcat_list.append(Xj)
          ycat_list.append(yj)

    Xcat = np.concatenate(Xcat_list)
   # print(Xcat.shape)
    ycat = np.concatenate(ycat_list)
    #print(ycat.shape)
    sel_pos = (ycat==sel_digit)
    Xpos = Xcat[sel_pos].copy()
    #print(Xpos.shape)
    ypos = np.ones(ycat[sel_pos].shape)
    sel_neg = (ycat!=sel_digit)
    Xneg = Xcat[sel_neg].copy()
    #print(Xneg.shape)
    yneg = np.zeros(ycat[sel_neg].shape)
    X = np.concatenate((Xpos,Xneg))
    y = np.concatenate((ypos,yneg))
    self.digit_dataset[sel_digit]=(X,y)


  def prepare_clf(self,sel_digit=None,central_model=None):
        print("prepare clf")
        central_model_weights=central_model.get_weights()
        X,y=self.digit_dataset[sel_digit]
        X, y = shuffle(X, y, random_state=0)
        y= to_categorical(y,2)
        local_model=self.digit_clf[sel_digit]
        local_model.set_weights(central_model_weights)
        epochs=5
       # local_model.compile(optimizer='SGD',loss= 'categorical_crossentropy', metrics=['accuracy'])
        local_model.fit(x=X, y=y, batch_size=15, epochs=epochs, verbose=0)
      # self.digit_dataset[sel_digit]=X,y
        self.digit_clf[sel_digit] = local_model

  def return_clf(self,sel_digit):

    return  self.digit_clf[sel_digit]

In [ ]:
# file create_device_and_init_model
def make_device(n_devices):
  device_list = []
  for i in range(n_devices):
    mydevice = MyDigitDevice(id=i)
  # print(getattr(mydevice,'id'))
  # break
    device_list.append(mydevice)
   # model_list.append(get_CNN_model())
  return device_list


## Build one-vs-rest classifier for each digit on each device

In [ ]:

def prepare_client_training_dataset_and_clf(device_list,label):
  print("Training dataset for the clients")
  label_device_map = { new_list: [] for new_list in range(len(label)) }

  #print (id_clf_map)

  for i in range(len(device_list)):
    print("training dataset prepared for client :" +str(i))
    my_device = device_list[i]
    client_label_list= list(my_device.digit_data.keys())
    for j in client_label_list:
      clf=get_CNN_model()
      my_device.digit_clf[j]=clf
      my_device.get_training_dataset(sel_digit=j)
      label_device_map[j].append(my_device)
  return label_device_map


# binarize

In [ ]:
# file 5  binarize
def prepare_binarize_dataset(yb,selected_digit):

    y_binarize=yb.copy()
    sel_pos=np.where(y_binarize==selected_digit)[0]

    sel_neg=np.where(y_binarize!=selected_digit)[0]

    for i in sel_pos:
      y_binarize[i]=1
    for i in sel_neg:
      y_binarize[i]=0

    return y_binarize



In [ ]:
def one_cycle_fedova(X_test,y_test,label_device_map):
    print("in voting")
    voting_clf=[]
    log_gm_final=[]
    log_odd_final=[]
    prob_yes_final=[]
    y_pred_gm=[]
    y_pred_odd=[]
    y_pred_yes=[]
    for k in range(0,len(label_device_map)):
      print(k)
      #estimators=[]
      #final_output=[]
      correct=0
      prob=[]
     # prob_yes=[]
      prob_yes_list=[]
      log_gm_list=[]
      log_odd_list=[]
      size=len(label_device_map[k])
      for j in range(0,len(label_device_map[k])):
          device=label_device_map[k][j]
          #print(device)
          #device.prepare_clf(j)
          model=device.digit_clf[k]
          prediction=model.predict(X_test)
          #prob_yes.append([np.argmax(prediction[m]) for m in range(0,X_test.shape[0])])
          prob.append(prediction)
        #  print(prob)

      for i in range(y_test.shape[0]):
              prob_1=[sublist[i] for sublist in prob]
             # prob_2=[sublist[i] for sublist in prob_yes]
              log_gm=np.sum([math.log(prob_1[m][1],2) for m in range(0,len(prob_1))])/size
              log_odd=np.sum([(math.log(prob_1[m][1],2)-(math.log(prob_1[m][0],2))) for m in range(0,len(prob_1))])/size
              yes=np.sum([np.argmax(prob_1[m]) for m in range(0,len(prob_1))])/size
              if i==1:
                print(yes)
             # print(log_gm)
             # print(log_odd)
              log_gm_list.append(log_gm)
              log_odd_list.append(log_odd)
              prob_yes_list.append(yes)
      log_gm_final.append(log_gm_list)
      log_odd_final.append(log_odd_list)
      prob_yes_final.append(prob_yes_list)
    for m in range(0,X_test.shape[0]):
        result_gm=[sublist[m] for sublist in log_gm_final]
        y_pred_gm.append(np.argmax(result_gm))
      #  print(result_gm)
        result_odd=[sublist[m] for sublist in log_odd_final]
        y_pred_odd.append(np.argmax(result_odd))
      #  print(result_odd)
      #  print(y_pred_odd)
        result_yes=[sublist[m] for sublist in prob_yes_final]
        y_pred_yes.append(np.argmax(result_yes))

    print(f"accuracy by voting mean log gm for bcc : {accuracy_score(y_test,y_pred_gm)*100}")
    print(f"recall by voting mean log gmm for bcc : {recall_score(y_test,y_pred_gm,average='macro')*100}")
    print(f"precision by voting mean log gm for bcc : {precision_score(y_test,y_pred_gm,average='macro')*100}")
    print("\n")

    print(f"accuracy by voting log of odds for bcc : {accuracy_score(y_test,y_pred_odd)*100}")
    print(f"recall by voting log of odds for bcc : {recall_score(y_test,y_pred_odd,average='macro')*100}")
    print(f"precision by voting log of odds for bcc : {precision_score(y_test,y_pred_odd,average='macro')*100}")
    print("\n")

    print(f"accuracy by voting by probablity of yes for bcc : {accuracy_score(y_test,y_pred_yes)*100}")
    print(f"recall by voting by probablity of yes for bcc : {recall_score(y_test,y_pred_yes,average='macro')*100}")
    print(f"precision by voting by probablity of yes for bcc : {precision_score(y_test,y_pred_yes,average='macro')*100}")
    print("\n")



# main

In [ ]:
n_devices=10
beta_value=2
parent_dir="/content/gdrive/MyDrive/dataset"


In [ ]:
X,y,X_test,y_test=load_dataset()



data loading
11490434/11490434 [==============================] - 0s 0us/step
(60000, 28, 28) (60000,) (10000, 28, 28) (10000,)


In [ ]:
X=data_processing(X)

data processing
(60000, 28, 28)


In [ ]:
X_test=data_processing(X_test)


data processing
(10000, 28, 28)


In [ ]:
labels=get_categories(y)

number of classes in the dataset: 10
classes in the dataset: [0 1 2 3 4 5 6 7 8 9]


In [ ]:
#make_dir(n_devices,parent_dir)

In [ ]:
digit_map=get_datamap(X,y,labels)


split data as per y label
(5923, 28, 28) (5923,)
(6742, 28, 28) (6742,)
(5958, 28, 28) (5958,)
(6131, 28, 28) (6131,)
(5842, 28, 28) (5842,)
(5421, 28, 28) (5421,)
(5918, 28, 28) (5918,)
(6265, 28, 28) (6265,)
(5851, 28, 28) (5851,)
(5949, 28, 28) (5949,)


In [ ]:
device_list=make_device(n_devices)

device id:0
device id:1
device id:2
device id:3
device id:4
device id:5
device id:6
device id:7
device id:8
device id:9


In [ ]:
device_list=noniid_distribuition(digit_map,device_list,labels,beta_value)

In non iid distribution part 1
dataset: 0
dataset: 1
dataset: 2
dataset: 3
dataset: 4
dataset: 5
dataset: 6
dataset: 7
dataset: 8
dataset: 9
In non iid distribution part 2


In [ ]:

#m=0
label_device_map=prepare_client_training_dataset_and_clf(device_list,labels)




Training dataset for the clients
training dataset prepared for client :0
in get dataset
in get dataset
training dataset prepared for client :1
in get dataset
in get dataset
training dataset prepared for client :2
in get dataset
in get dataset
training dataset prepared for client :3
in get dataset
in get dataset
training dataset prepared for client :4
in get dataset
in get dataset
training dataset prepared for client :5
in get dataset
in get dataset
training dataset prepared for client :6
in get dataset
in get dataset
training dataset prepared for client :7
in get dataset
in get dataset
training dataset prepared for client :8
in get dataset
in get dataset
training dataset prepared for client :9
in get dataset
in get dataset


In [ ]:
one_cycle_fedova(,X_test,y_test,label_device_map)